In [21]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.nn.utils.prune as prune

from tensorboardX import SummaryWriter      

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.append('..')
from models import *

import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant"
model = VGG16_quant()
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [36]:
model = VGG16_quant()
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

def check_accuracy(model):
    model.cuda()
    model.eval()
    
    test_loss = 0
    correct = 0
    
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device) # loading to GPU
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)  
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(testloader.dataset)
    
    print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
            correct, len(testloader.dataset),
            100. * correct / len(testloader.dataset)))

check_accuracy(model)


Test set: Accuracy: 8434/10000 (84%)



## Unstructured pruning

In [37]:
#### Prune all the QuantConv2D layers' 90% weights with 1) unstructured, and 2) structured manner.
def prune_model_unstructured(model, amount=0.8):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=amount)
    return model

model = prune_model_unstructured(model, 0.8)

In [ ]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

In [38]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

Sparsity level:  tensor(0.8000, device='cuda:0')


In [39]:
def print_per_layer_sparsity(model):
    total_zeros, total_params = 0, 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            w = module.weight
            zeros = (w == 0).sum().item()
            n = w.numel()
            total_zeros += zeros
            total_params += n
            print(f"{name:30s} {100 * zeros / n:6.2f}%  ({zeros}/{n})")
    print(f"{'Overall (conv layers)':30s} {100 * total_zeros / total_params:6.2f}%")

print_per_layer_sparsity(model)

features.0                      79.98%  (1382/1728)
features.3                      80.00%  (29491/36864)
features.7                      80.00%  (58982/73728)
features.10                     80.00%  (117965/147456)
features.14                     80.00%  (235930/294912)
features.17                     80.00%  (471859/589824)
features.20                     80.00%  (471859/589824)
features.24                     80.00%  (943718/1179648)
features.27                     80.00%  (1887437/2359296)
features.30                     80.00%  (1887437/2359296)
features.34                     80.00%  (1887437/2359296)
features.37                     80.00%  (1887437/2359296)
features.40                     80.00%  (1887437/2359296)
Overall (conv layers)           80.00%


In [40]:
## check accuracy after pruning
check_accuracy(model)


Test set: Accuracy: 1000/10000 (10%)



In [ ]:
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr ##
# This cell is from the website

lr = 4e-3

criterion = nn.CrossEntropyLoss().cuda()

weight_decay = 1e-4
epochs = 30
best_prec = validate(testloader, model, criterion) # 0


model = model.cuda()

#optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
# weight decay: for regularization to prevent overfitting

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)



if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)+'_pruned'

if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    
    #adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    scheduler.step()
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)
    


In [29]:
## check your accuracy again after finetuning
check_accuracy(model)
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)


Test set: Accuracy: 8758/10000 (88%)

Sparsity level:  tensor(0.8000, device='cuda:0')


## Structured pruning

In [41]:
# structured pruning
def prune_model_structured(model, amount=0.8, n=2, dim=0):
    """
    Structured pruning on every Conv2d layer (in place).
    dim=0 prunes output filters, dim=1 prunes input channels.
    n=2 ranks by L2 norm, n=1 by L1 norm.
    """
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name="weight", amount=amount, n=n, dim=dim)
    return model

model = VGG16_quant()
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

prune_model_structured(model)

check_accuracy(model)
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)
print_per_layer_sparsity(model)


Test set: Accuracy: 1000/10000 (10%)

Sparsity level:  tensor(0.8008, device='cuda:0')
features.0                      79.69%  (1377/1728)
features.3                      79.69%  (29376/36864)
features.7                      79.69%  (58752/73728)
features.10                     79.69%  (117504/147456)
features.14                     80.08%  (236160/294912)
features.17                     80.08%  (472320/589824)
features.20                     80.08%  (472320/589824)
features.24                     80.08%  (944640/1179648)
features.27                     80.08%  (1889280/2359296)
features.30                     80.08%  (1889280/2359296)
features.34                     80.08%  (1889280/2359296)
features.37                     80.08%  (1889280/2359296)
features.40                     80.08%  (1889280/2359296)
Overall (conv layers)           80.07%


In [ ]:
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr ##
# This cell is from the website

lr = 4e-3

criterion = nn.CrossEntropyLoss().cuda()

weight_decay = 1e-4
epochs = 30
best_prec = validate(testloader, model, criterion) # 0


model = model.cuda()

#optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
# weight decay: for regularization to prevent overfitting

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)



if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)+'_pruned_structured'

if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    
    #adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    scheduler.step()
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)
    


In [34]:
## check your accuracy again after finetuning
check_accuracy(model)
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)


Test set: Accuracy: 8165/10000 (82%)

Sparsity level:  tensor(0.8008, device='cuda:0')


## Pruning Comparison

| Method | Pruning amount | Test acc. (dense, before pruning) | Test acc. right after pruning | Test acc. after fine-tuning | 
|---|---|---|---|---|
| Unstructured  | 0.8 | 84% | 10% | 88% | 
| Structured  | 0.8 | 84% | 10% | 82% | 



The structured pruning recovered less accuracy after fine tuning. This makes sense since structured pruning gets rid of entire output filters while unstructured only removes weights. So the structured pruning will have less things to modify while training so it won't be able to increase accuracy as much.